# Interazione con HDFS tramite Python
Questo notebook mostra come interagire con un cluster Apache HDFS tramite l'interfaccia WebHDFS utilizzando la libreria `hdfs` in Python.

### Operazioni eseguite:
1. Connessione a HDFS NameNode
2. Caricamento del dataset locale su HDFS
3. Verifica della presenza del file e lettura dei metadati
4. Download del dataset da HDFS salvandolo con un nome differente

In [1]:
import os
from hdfs import InsecureClient

# Connessione al NameNode (WebHDFS porta 9870)
client = InsecureClient('http://namenode:9870', user='hadoop')
print('Root directory status:', client.status('/'))

Root directory status: {'accessTime': 0, 'blockSize': 0, 'childrenNum': 0, 'fileId': 16385, 'group': 'supergroup', 'length': 0, 'modificationTime': 0, 'owner': 'hadoop', 'pathSuffix': '', 'permission': '755', 'replication': 0, 'snapshotEnabled': True, 'storagePolicy': 0, 'type': 'DIRECTORY'}


### 1. File locale del Dataset
Il dataset è montato nel volume locale in `/app/datasets`.

In [2]:
local_dataset_path = '/app/datasets/Solar_Energy_Production.csv'
hdfs_dir = '/datasets'
hdfs_file_path = f'{hdfs_dir}/Solar_Energy_Production.csv'

print(f'File locale: {local_dataset_path}')
print(f'Dimensione locale: {os.path.getsize(local_dataset_path):,} bytes')

File locale: /app/datasets/Solar_Energy_Production.csv
Dimensione locale: 63,032,168 bytes


### 2. Caricamento del file su HDFS

In [3]:
# Creazione directory se non presente
client.makedirs(hdfs_dir)

# Caricamento file
print(f'Caricamento di {local_dataset_path} su {hdfs_file_path}...')
client.upload(hdfs_file_path, local_dataset_path, overwrite=True)
print('Upload completato con successo!')

Caricamento di /app/datasets/Solar_Energy_Production.csv su /datasets/Solar_Energy_Production.csv...
Upload completato con successo!


### 3. Verifica presenza del file su HDFS e ispezione metadati

In [4]:
# Elenco file nella directory HDFS
hdfs_files = client.list(hdfs_dir)
print(f'File presenti in {hdfs_dir}:', hdfs_files)
assert 'Solar_Energy_Production.csv' in hdfs_files, 'File non trovato su HDFS!'

# Ispezione metadati HDFS
metadata = client.status(hdfs_file_path)
print('\nMetadati del file su HDFS:')
for key, val in metadata.items():
    print(f'  {key}: {val}')

File presenti in /datasets: ['Solar_Energy_Production.csv']

Metadati del file su HDFS:
  accessTime: 1790763914073
  blockSize: 134217728
  childrenNum: 0
  fileId: 16387
  group: supergroup
  length: 63032168
  modificationTime: 1790763917005
  owner: hadoop
  pathSuffix: 
  permission: 644
  replication: 2
  storagePolicy: 0
  type: FILE


### 4. Download da HDFS con un nome differente

In [5]:
download_file_path = '/app/notebooks/Solar_Energy_Production_renamed.csv'

# Rimuoviamo il file se precedentemente scaricato
if os.path.exists(download_file_path):
    os.remove(download_file_path)

print(f'Download in corso da HDFS ({hdfs_file_path}) a {download_file_path}...')
client.download(hdfs_file_path, download_file_path)
print('Download completato!')

# Verifica file scaricato
print(f'Dimensione file scaricato: {os.path.getsize(download_file_path):,} bytes')
assert os.path.getsize(download_file_path) == os.path.getsize(local_dataset_path), 'Dimensione diversa dal dataset originale!'

Download in corso da HDFS (/datasets/Solar_Energy_Production.csv) a /app/notebooks/Solar_Energy_Production_renamed.csv...
Download completato!
Dimensione file scaricato: 63,032,168 bytes
